# 06. Streaming Sentiment Scoring

We analyzed live comments from the God of War Laufey video: [https://www.youtube.com/watch?v=022NmtXxPyk](https://www.youtube.com/watch?v=022NmtXxPyk).

The streaming producer that sends comments to Kafka is located in `infra/youtube_producer/producer.py`.

This notebook runs near real-time sentiment scoring on incoming YouTube comments using Spark Structured Streaming and stores scored outputs in MongoDB.

## 1. Environment Setup

In [1]:
import os
import pandas as pd
from pathlib import Path

from kafka import KafkaAdminClient
from pymongo import MongoClient, ReplaceOne
from pyspark.ml import PipelineModel
from pyspark.ml.feature import StringIndexerModel
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructField, StructType, StringType


## 2. Runtime Configuration

This section defines infrastructure and model paths for the streaming job.

Note on `TS_IP_ADDRESS`: this IP is not a secret credential. In this project it is acceptable to commit because authentication is enforced by the tailnet, and the endpoint is only reachable by users inside the tailnet.

In [2]:
# --- Runtime configuration (clean + minimal) ---
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()

# Infra connection
MONGO_USER = os.getenv('MONGO_USER', 'admin')
MONGO_PASS = os.getenv('MONGO_PASS', 'supersecret')
MONGO_HOST = os.getenv('TS_IP_ADDRESS', '100.95.107.113').split('#')[0].strip()
MONGO_PORT = int(os.getenv('MONGO_PORT', '27017'))
KAFKA_TOPIC = os.getenv('KAFKA_TOPIC', 'youtube_comments')
KAFKA_BROKER = os.getenv('KAFKA_BROKER', f"{MONGO_HOST}:9092")

# Use the best model produced by notebooks/all_models.ipynb
MODEL_PATH = os.getenv(
    'SENTIMENT_MODEL_PATH',
    str(PROJECT_ROOT / 'data' / 'youtube-comment-sentiment' / 'youtube_sentiment_svm_(ovr)_pipeline')
)

# Streaming behavior
MONITOR_SECONDS = int(os.getenv('MONITOR_SECONDS', '3600'))  # 1 hour by default
TRIGGER_SECONDS = int(os.getenv('TRIGGER_SECONDS', '30'))
CHECKPOINT_PATH = os.getenv(
    'STREAMING_CHECKPOINT_PATH',
    str(PROJECT_ROOT / 'data' / 'processed' / 'checkpoints' / 'youtube_scoring')
)

# Output collection
MONGO_DB_NAME = os.getenv('MONGO_DB_NAME', 'youtube_stream')
MONGO_RESULTS_COLLECTION = os.getenv('MONGO_RESULTS_COLLECTION', 'scored_comments')

mongo_uri = f"mongodb://{MONGO_USER}:{MONGO_PASS}@{MONGO_HOST}:{MONGO_PORT}/?authSource=admin"

# --- Single persistent MongoDB client (used throughout) ---
mongo_client = MongoClient(mongo_uri, serverSelectionTimeoutMS=5000)

print('Model path:', MODEL_PATH)
print('Kafka broker:', KAFKA_BROKER)
print('Kafka topic:', KAFKA_TOPIC)
print('Monitor seconds:', MONITOR_SECONDS)


Model path: /home/lucas/Desktop/Big-Data-Analyitics-Project/data/youtube-comment-sentiment/youtube_sentiment_svm_(ovr)_pipeline
Kafka broker: 100.95.107.113:9092
Kafka topic: youtube_comments
Monitor seconds: 3600


## 3. Connectivity Checks

In [3]:
# --- Quick connectivity checks ---
mongo_client.admin.command('ping')

admin = KafkaAdminClient(bootstrap_servers=KAFKA_BROKER, client_id='infra-check')
topics = set(admin.list_topics())
admin.close()

if KAFKA_TOPIC not in topics:
    raise ValueError(f"Kafka topic '{KAFKA_TOPIC}' not found. Available topics: {sorted(topics)}")

print('MongoDB and Kafka are reachable.')


MongoDB and Kafka are reachable.


## 4. Streaming Inference Pipeline

The best-performing model was selected from `02_ml_classification.ipynb` (Linear SVM OvR pipeline).

We use Spark MLlib for inference because it integrates natively with Structured Streaming, reducing integration overhead and improving end-to-end inference time in this streaming workflow.

### Real-Time Scoring Flow

1. Read new YouTube comments continuously from Kafka using the infra message schema.
2. Apply the model selected in `02_ml_classification.ipynb` using Spark MLlib stages.
3. Process data as a stream, but persist predictions in micro-batches to MongoDB via `foreachBatch` + bulk upserts.
4. Keep the query active for the monitoring window, then inspect sentiment distribution from stored records.

In [4]:
# --- Spark session ---
try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('youtube-streaming-scoring')
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.2')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print('Spark ready:', spark.version)

:: loading settings :: url = jar:file:/home/lucas/Desktop/Big-Data-Analyitics-Project/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/lucas/.ivy2.5.2/cache
The jars for the packages stored in: /home/lucas/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-45892ec8-d966-4cbd-9f74-425c678a9187;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.2 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found com.google.code.findbugs

Spark ready: 4.1.2


In [5]:
# --- Input schema must match infra producer payload exactly ---
schema = StructType([
    StructField('comment_id', StringType(), True),
    StructField('video_id', StringType(), True),
    StructField('text', StringType(), True),
    StructField('author', StringType(), True),
    StructField('published_at', StringType(), True),
    StructField('fetched_at', StringType(), True),
    StructField('source', StringType(), True),
])

# Load trained pipeline from all_models output
model = PipelineModel.load(MODEL_PATH)

# Extract label names from StringIndexer (for readable sentiment output)
indexer_stage = next((s for s in model.stages if isinstance(s, StringIndexerModel)), None)
label_values = []
if indexer_stage is not None:
    if hasattr(indexer_stage, 'labels') and indexer_stage.labels:
        label_values = list(indexer_stage.labels)
    elif hasattr(indexer_stage, 'labelsArray') and indexer_stage.labelsArray:
        label_values = list(indexer_stage.labelsArray[0])

# Read Kafka stream (raw records)
raw_stream = (
    spark.readStream
    .format('kafka')
    .option('kafka.bootstrap.servers', KAFKA_BROKER)
    .option('subscribe', KAFKA_TOPIC)
    .option('startingOffsets', 'earliest')
    .load()
)

def score_and_upsert_batch(batch_df, batch_id):
    """Score one micro-batch with the trained model and upsert results to MongoDB."""
    parsed_df = (
        batch_df
        .select(F.from_json(F.col('value').cast('string'), schema).alias('payload'))
        .select('payload.*')
    )

    if parsed_df.rdd.isEmpty():
        print(f'Batch {batch_id}: 0 rows')
        return

    # Adapt infra stream columns to model input contract
    scored_input_df = parsed_df.withColumn('clean_text', F.coalesce(F.col('text'), F.lit('')))
    if label_values:
        # Some saved pipelines include StringIndexer; provide a placeholder label column
        scored_input_df = scored_input_df.withColumn('Sentiment', F.lit(label_values[0]))

    # Apply all stages except StringIndexerModel (training-only)
    scored_df = scored_input_df
    for stage in model.stages:
        if isinstance(stage, StringIndexerModel):
            continue
        scored_df = stage.transform(scored_df)

    # Convert numeric prediction index back to readable sentiment label
    if label_values:
        prediction_map = F.create_map(*[x for i, lbl in enumerate(label_values) for x in (F.lit(float(i)), F.lit(lbl))])
        scored_df = scored_df.withColumn('predicted_sentiment', prediction_map[F.col('prediction')])
    else:
        scored_df = scored_df.withColumn('predicted_sentiment', F.col('prediction').cast('string'))

    result_df = scored_df.select(
        'comment_id',
        'video_id',
        'author',
        'text',
        'published_at',
        'fetched_at',
        'source',
        'predicted_sentiment',
        F.col('prediction').cast('int').alias('prediction_index'),
        F.lit(Path(MODEL_PATH).name).alias('model_name'),
        F.current_timestamp().alias('scored_at')
    )

    docs = [row.asDict(recursive=True) for row in result_df.collect()]
    if not docs:
        print(f'Batch {batch_id}: 0 scored rows')
        return

    collection = mongo_client[MONGO_DB_NAME][MONGO_RESULTS_COLLECTION]
    collection.create_index([('comment_id', 1), ('video_id', 1)], unique=True)

    ops = [
        ReplaceOne(
            {'comment_id': d['comment_id'], 'video_id': d['video_id']},
            d,
            upsert=True,
        )
        for d in docs
    ]
    write_result = collection.bulk_write(ops, ordered=False)

    print(
        f"Batch {batch_id}: rows={len(docs)}, "
        f"matched={write_result.matched_count}, "
        f"modified={write_result.modified_count}, "
        f"upserted={len(write_result.upserted_ids)}"
    )

query = (
    raw_stream.writeStream
    .foreachBatch(score_and_upsert_batch)
    .outputMode('append')
    .option('checkpointLocation', CHECKPOINT_PATH)
    .trigger(processingTime=f'{TRIGGER_SECONDS} seconds')
    .start()
)

print(f'Running stream for {MONITOR_SECONDS} seconds...')
query.awaitTermination(MONITOR_SECONDS)

if query.isActive:
    query.stop()
    print('Stream stopped after monitoring window.')
else:
    print('Stream finished before monitoring window ended.')

# Quick verification from MongoDB
collection = mongo_client[MONGO_DB_NAME][MONGO_RESULTS_COLLECTION]
print('Total scored docs in MongoDB:', collection.count_documents({}))


Running stream for 3600 seconds...


Batch 0: rows=48, matched=0, modified=0, upserted=48
Batch 1: rows=3, matched=0, modified=0, upserted=3
Batch 2: rows=2, matched=0, modified=0, upserted=2
Batch 3: rows=1, matched=0, modified=0, upserted=1
Batch 4: rows=1, matched=0, modified=0, upserted=1


Batch 5: rows=1, matched=0, modified=0, upserted=1
Batch 6: rows=1, matched=0, modified=0, upserted=1
Batch 7: rows=1, matched=0, modified=0, upserted=1
Batch 8: rows=1, matched=0, modified=0, upserted=1
Batch 9: rows=1, matched=0, modified=0, upserted=1
Batch 10: rows=1, matched=0, modified=0, upserted=1
Batch 11: rows=1, matched=0, modified=0, upserted=1
Batch 12: rows=1, matched=0, modified=0, upserted=1
Batch 13: rows=1, matched=0, modified=0, upserted=1
Batch 14: rows=1, matched=0, modified=0, upserted=1
Batch 15: rows=1, matched=0, modified=0, upserted=1
Batch 16: rows=1, matched=0, modified=0, upserted=1
Batch 17: rows=1, matched=0, modified=0, upserted=1
Stream stopped after monitoring window.
Total scored docs in MongoDB: 68


## 5. Results Inspection

Review a recent sample of scored comments and the sentiment distribution stored in MongoDB.

In [6]:
_col = mongo_client[MONGO_DB_NAME][MONGO_RESULTS_COLLECTION]

docs = list(_col.find({}, {'_id': 0}).sort('scored_at', -1).limit(200))

if not docs:
    print('No documents in MongoDB yet.')
else:
    df = pd.DataFrame(docs)
    display_cols = [c for c in ['comment_id', 'video_id', 'author', 'text', 'predicted_sentiment', 'prediction_index', 'scored_at', 'model_name'] if c in df.columns]
    df = df[display_cols]

    print(f"Total docs shown: {len(df)}\n")
    print("Sentiment distribution:")
    print(df['predicted_sentiment'].value_counts().to_string())
    print()
    pd.set_option('display.max_colwidth', 80)
    display(df)


Total docs shown: 68

Sentiment distribution:
predicted_sentiment
Neutral     37
Positive    17
Negative    14



,comment_id,video_id,author,text,predicted_sentiment,prediction_index,scored_at,model_name
0,UgwyFiMosePRr-yv82V4AaABAg,022NmtXxPyk,@assinosombra,Cadê o Kratos,Neutral,2,2026-06-03 17:41:30.786,youtube_sentiment_svm_(ovr)_pipeline
1,Ugy7gBUetfLOGL02ROV4AaABAg,022NmtXxPyk,@86MissDani,Lindeza de jogo! 😍 Uma bela surpresa!,Neutral,2,2026-06-03 17:39:30.797,youtube_sentiment_svm_(ovr)_pipeline
2,Ugz62hVU2F7c2IRTPxB4AaABAg,022NmtXxPyk,@brunorocha7572,18:22 sempre quis fazer isso com alguem,Negative,0,2026-06-03 17:33:30.802,youtube_sentiment_svm_(ovr)_pipeline
3,Ugx46IxjSqfhgiHzSIl4AaABAg,022NmtXxPyk,@JoãoMarcoOo,"Ultimo God of War bom foi o 3, abraços",Neutral,2,2026-06-03 17:31:00.801,youtube_sentiment_svm_(ovr)_pipeline
4,UgzMucZI9J-C0dFnIQV4AaABAg,022NmtXxPyk,@Cristãodochat,"A gameplay parece boa, gráfico bom e direção interessante, mas... não é mais...",Positive,1,2026-06-03 17:28:00.833,youtube_sentiment_svm_(ovr)_pipeline
...,...,...,...,...,...,...,...,...
63,UgwIw7bEQnjmooJxlyB4AaABAg,022NmtXxPyk,@Daniel-i1s-e2d,"Espero que o jogo não flope, tem páginas falando mal outras falando bem, que...",Positive,1,2026-06-03 16:48:59.636,youtube_sentiment_svm_(ovr)_pipeline
64,Ugz_ihYM2XGEs8n5WH54AaABAg,022NmtXxPyk,@giulianocorreaart,Debora Ann Woll ❤,Neutral,2,2026-06-03 16:48:59.636,youtube_sentiment_svm_(ovr)_pipeline
65,UgzZI_RoYUI2EMsROA14AaABAg,022NmtXxPyk,@andreapessoa7124,UHUUUUUUUUUUU!!! TOP!!! 🤩,Positive,1,2026-06-03 16:48:59.636,youtube_sentiment_svm_(ovr)_pipeline
66,UgxjauMCxAA-mvhnVyx4AaABAg,022NmtXxPyk,@pedrogamadm,Lixo,Neutral,2,2026-06-03 16:48:59.636,youtube_sentiment_svm_(ovr)_pipeline


## 6. Cleanup

Close open clients cleanly after the streaming window ends.

In [7]:
# --- Cleanup: Close MongoDB client ---
mongo_client.close()
print("MongoDB client closed.")


MongoDB client closed.


## 7. Conclusion

This streaming pipeline is effective for real-time sentiment scoring and operational monitoring, but Spark MLlib also introduces practical limits in this context:

- The current linear model can miss nuanced language (sarcasm, slang shifts, mixed sentiment, domain-specific context).
- Language coverage is limited in multilingual comments. For example, a short comment like `Lixo` (Portuguese for `garbage`) can be incorrectly scored as `Neutral` instead of `Negative`.
- Feature engineering in classic ML pipelines is less expressive than modern contextual embeddings.
- Structured Streaming + micro-batch scoring is robust, but latency and throughput trade-offs can still affect online quality.

For future iterations, we can improve prediction accuracy by replacing the current MLlib classifier with a stronger model (for example, a fine-tuned transformer-based sentiment model) while preserving the same streaming architecture for ingestion and persistence.